In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph , START ,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv


In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [4]:
class subState(TypedDict):
    input_text:str
    translated_text:str

In [5]:
subLLm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [6]:
def translate_text(state: subState):
    propmt = f"""
Translate the following text to hindi keep it natural and clear do not add extra content

Text: {state['input_text']}
""".strip()
    translate_text = subLLm.invoke(propmt)
    return {'translate_text' : translate_text}                                   


In [7]:
subgraph_build = StateGraph(subState)

subgraph_build.add_node('translate_text' , translate_text)

subgraph_build.add_edge(START , 'translate_text')
subgraph_build.add_edge('translate_text',END)

subgraph = subgraph_build.compile()

In [8]:
class ParentState(TypedDict):
    question:str
    answer_eng:str
    answer_hin:str

In [9]:
parent_llm = ChatGoogleGenerativeAI(model = "gemini-3.6-flash")

In [10]:
def generate_answer(state: ParentState):

    result = parent_llm.invoke(f"you are helpful assistant answer cleraly Questions . {state['question']}")

    return {'answer_eng' : result.text}

In [11]:
def  generate_hindi(state:ParentState):
    result = subgraph.invoke({'input_text':{state['answer_eng']}})

    return {'answer_hin':result}

In [12]:
parent_build  = StateGraph(ParentState)

parent_build.add_node('generate_answer',generate_answer)
parent_build.add_node('generate_hindi',generate_hindi)

parent_build.add_edge(START , 'generate_answer')
parent_build.add_edge('generate_answer','generate_hindi')
parent_build.add_edge('generate_hindi',END)

parent_workFlow = parent_build.compile()

In [13]:
prompt =   {'question':'what is Quantum physics explain in 100 -200 words '}


In [14]:
response = parent_workFlow.invoke(prompt)
print(response)

{'question': 'what is Quantum physics explain in 100 -200 words ', 'answer_eng': '**Quantum physics** (or quantum mechanics) is the branch of science that studies matter and energy at the smallest possible scale—specifically atoms and subatomic particles like electrons and photons. \n\nAt this microscopic level, the usual laws of classical physics no longer apply. Instead, the universe behaves in strange and counterintuitive ways:\n\n* **Wave-Particle Duality:** Particles can act like solid objects in one moment and like spreading waves in the next.\n* **Superposition:** A particle can exist in multiple states or locations at the exact same time until it is measured.\n* **Quantum Entanglement:** Two particles can become deeply connected, so that changing one instantly affects the other, no matter how far apart they are.\n\nQuantum physics is not just theoretical; it is the foundation of much of our modern technology. Without quantum mechanics, we would not have lasers, LED lights, sola